# Notebook 8 — Stress test, frozen re-test of spectral v0.5, and the cost of evasion
Two phases in one session (≈ 2.5–3 h on a T4).

**Phase A — stress (60 min).**
- Prompt and output lengths are drawn from WildChat (up to 1,024 output tokens), not a fixed 96 tokens.
- vLLM's KV cache is squeezed so the server preempts sequences.
- Humans use WildChat think times and session lengths.
- Populations are as in NB7. The answer end carries the NB7 hold-back watermark (0 or 150 ms).
- Every detector runs **frozen**: classifier thresholds from simulation, sideband graph (after NB6) and spectral rings **v0.5** (after NB7).

**Phase B — evasion gradient (40 min).** NB7-style short answers. Twelve attacker variants on 20 keys each, including jitter, keys held for H requests, workers near the number of keys, pinned keys, and steady and bursty open loops. The watermark is **spread** over the stream instead of held at the end.

**Locked predictions** (scored at the end):
- **A0** the stress is real: KV-cache usage reaches ≥ 0.85, or vLLM reports preemptions.
- **A1** frozen classifier (v0.5, 20 s high-pass) ≥ 10 of 12 pools correct. *Revised after the dry run on the simulated server, before any vLLM data:* with long answers a ring's period T_c = M/λ can exceed the 20 s block, which then removes the ring. So we expect v0.5 to **miss rings with T_c > 20 s**, and the A1 count may fail for that reason.
- **A1b** multi-scale classifier (v0.6: ring statistic at 20, 60 and 180 s blocks, defined after the dry run and before this run) ≥ 10 of 12 correct.
- **A2** spectral v0.5: P3, P5 and P8 each have recall ≥ 0.9 and purity ≥ 0.9; at most 2 humans selected (expected to fail for rings with T_c > 20 s).
- **A2b** spectral v0.6 (union over 20, 60 and 180 s blocks): P3, P5 and P8 recall ≥ 0.9; at most 2 humans.
- **A3** sideband graph: P3 and P4 recall ≥ 0.9; at most 2 humans.
- **A4** random relay P7: no cross-key echo (p > 0.01), replicating NB7's anomaly as chance.
- **A5** nulls hold under stress: ≤ 2 of 50 human pools at watermark p < 0.01, and ≤ 5% of human pools at |z_R| > 3.
- **A6** worker count (interactive response-time law) within 25% for P1, P2, P3, P4 and C.
- **B1** ring z > 6 for the shared counter at jitter 0, 2, 6 and 15 s, and with 24 workers at 15 s.
- **B2** ring z > 3 when workers hold a key for 5 requests; with 20 requests the pool is still flagged by turn-taking (z_R > 3).
- **B3** classification (v0.6) ≥ 10 of 12 phase-B variants correct (alternatives listed per pool).
- **B4** spread watermark: cross-key p < 0.001 for E0, E7 and E8; p > 0.001 for E2 and E3, as the jitter model predicts.
- **B5** visibility: the coin is readable from the last inter-chunk gap with the hold-back (AUC ≥ 0.9, phase A) but not with the spread delay (AUC ≤ 0.6, phase B).
- **B6** pinned keys (E9) and the steady open loop (E10): class none, cross-key echo p > 0.01.
- **B7** the bursty open loop (E11) is flagged as bunching (z_R < −3).

**Run settings.** Accelerator **GPU T4 ×1**, Internet **On**. Paste back the PASTE block and upload `nb8_requests_A.csv` and `nb8_requests_B.csv`.

In [ ]:
# Cell 1 — settings
import os, sys, json, time, subprocess, signal, glob, threading, urllib.request
T_START = time.time()
NH = int(os.environ.get("NB8_HUMANS", 400))
HUM = lambda: {"hum_a": dict(kind="human", n=NH // 2, key_offset=0), "hum_b": dict(kind="human", n=NH - NH // 2, key_offset=NH // 2),
               "bat": dict(kind="batch", n=4)}
CFG = dict(
    backend=os.environ.get("NB8_BACKEND", "vllm"), model="Qwen/Qwen2.5-0.5B-Instruct", seed=8,
    run_s={"A": int(os.environ.get("NB8_RUN_A", 3600)), "B": int(os.environ.get("NB8_RUN_B", 2400))},
    warmup_s=int(os.environ.get("NB8_WARMUP", 240)),
    tok_med=96, tok_sig=0.6, client_gap_med=0.2, max_out=1024, max_prompt=1500,
    human=dict(gap_s=float(os.environ.get("NB8_HUMAN_GAP", 360)), mean_req=6, think_med_s=6.0, think_sig=0.9),
    agent=dict(mean_steps=8, tool_med_s=2.0, tok_med=64),
    lengths={"A": "wildchat", "B": "fixed"}, humans="wildchat",
    wm_mode={"A": "holdback", "B": "spread"}, wm_delta=0.15,
    vllm_extra={"A": [["--num-gpu-blocks-override", "1600"], ["--gpu-memory-utilization", "0.12"]], "B": [[]]},
    pops={
        "A": dict(HUM(),
                  P1_shuffle=dict(kind="litellm", strategy="simple-shuffle", K=6, M=40),
                  P2_least=dict(kind="litellm", strategy="least-busy", K=10, M=40),
                  P3_rrshared=dict(kind="rr_shared", K=12, M=40), P4_rrworker=dict(kind="rr_worker", K=8, M=40),
                  P5_rrjitter=dict(kind="rr_shared", K=8, M=40, jitter=6.0), P6_agents=dict(kind="agents", K=6, M=32),
                  P7_relay_random=dict(kind="open_loop", lam=1.0, policy="random", M=40),
                  P8_relay_rr=dict(kind="open_loop", lam=1.0, policy="rr", M=40),
                  C_company=dict(kind="litellm", strategy="simple-shuffle", K=4, M=10), P9_pinned=dict(kind="pinned", K=8, M=8)),
        "B": dict(HUM(),
                  E0_rr_J0=dict(kind="rr_shared", K=8, M=20), E1_rr_J2=dict(kind="rr_shared", K=8, M=20, jitter=2.0),
                  E2_rr_J6=dict(kind="rr_shared", K=8, M=20, jitter=6.0), E3_rr_J15=dict(kind="rr_shared", K=8, M=20, jitter=15.0),
                  E4_rr_J15_K24=dict(kind="rr_shared", K=24, M=20, jitter=15.0),
                  E5_hold5=dict(kind="rr_hold", K=8, M=20, H=5), E6_hold20=dict(kind="rr_hold", K=8, M=20, H=20),
                  E7_shuffle_K8=dict(kind="random", K=8, M=20), E8_shuffle_K18=dict(kind="random", K=18, M=20),
                  E9_pinned=dict(kind="pinned", K=8, M=8),
                  E10_open_steady=dict(kind="open_loop", lam=3.0, policy="random", M=20),
                  E11_open_bursty=dict(kind="open_loop", lam=6.0, policy="random", M=20, on_s=30.0, off_s=60.0)),
    },
    expected={
        "A": {"hum_random40": "none", "benign_36hum_4bat": "none", "P1_shuffle": "random / shuffle", "P2_least": "sticky / episodic",
              "P3_rrshared": "shared round-robin (closed loop)", "P4_rrworker": "per-worker round-robin",
              "P5_rrjitter": "shared round-robin (closed loop)", "P6_agents": "sticky / episodic", "P7_relay_random": "none",
              "P8_relay_rr": "shared round-robin (open-loop relay)", "C_company": "random / shuffle", "P9_pinned": "none"},
        "B": {"hum_random40": "none", "E0_rr_J0": "shared round-robin (closed loop)", "E1_rr_J2": "shared round-robin (closed loop)",
              "E2_rr_J6": "shared round-robin (closed loop)", "E3_rr_J15": "shared round-robin (closed loop)",
              "E4_rr_J15_K24": "shared round-robin (closed loop) | shared round-robin (open-loop relay)",
              "E5_hold5": "shared round-robin (closed loop) | sticky / episodic", "E6_hold20": "shared round-robin (closed loop) | sticky / episodic",
              "E7_shuffle_K8": "random / shuffle", "E8_shuffle_K18": "random / shuffle", "E9_pinned": "none", "E10_open_steady": "none",
              "E11_open_bursty": "bunching"},
    },
    gateway="http://127.0.0.1:9000",
)
print(json.dumps({k: v for k, v in CFG.items() if k not in ("pops", "expected")}, indent=1))


In [ ]:
%%writefile occlib.py
"""
occlib — occupancy statistics for detecting hidden closed-loop worker pools behind many accounts.

Data model: a request log as three equal-length arrays
    acct : int   account index 0..A-1
    s    : float request arrival time (seconds)
    e    : float time the streamed response finished (seconds), e >= s
X_i(t) = number of account i's requests in flight; N(t) = sum_i X_i(t).

All statistics are exact time integrals of step functions (no time grid).
"""
import numpy as np

__version__ = "0.1.0"

# --------------------------------------------------------------------------------------------
# core: exact per-group integrals of X and X^2
# --------------------------------------------------------------------------------------------
def _group_integrals(gid, s, e, ngroups):
    """For intervals carrying a group id, return I1[g] = int X_g dt and I2[g] = int X_g^2 dt,
    where X_g(t) = number of the group's intervals covering t (half-open [s, e))."""
    I1 = np.zeros(ngroups); I2 = np.zeros(ngroups)
    n = len(s)
    if n == 0:
        return I1, I2
    t = np.concatenate([s, e])
    d = np.concatenate([np.ones(n), -np.ones(n)])
    g = np.concatenate([gid, gid])
    o = np.lexsort((d, t, g))          # by group, then time, then ends before starts
    t, d, g = t[o], d[o], g[o]
    x = np.cumsum(d)                    # each group nets to zero, so the level resets between groups
    dt = np.empty_like(t)
    dt[:-1] = t[1:] - t[:-1]
    dt[-1] = 0.0
    dt[:-1][g[1:] != g[:-1]] = 0.0
    I1 += np.bincount(g, weights=x * dt, minlength=ngroups)
    I2 += np.bincount(g, weights=x * x * dt, minlength=ngroups)
    return I1, I2


def _clip(acct, s, e, t0, t1):
    s = np.clip(np.asarray(s, float), t0, t1); e = np.clip(np.asarray(e, float), t0, t1)
    m = e > s
    return np.asarray(acct)[m], s[m], e[m]


def _split_blocks(acct, s, e, t0, tau, nb):
    """Split intervals at block boundaries t0 + k*tau. Returns (acct, block, s, e) pieces."""
    b0 = np.floor((s - t0) / tau).astype(np.int64)
    b1 = np.floor((np.nextafter(e, -np.inf) - t0) / tau).astype(np.int64)
    b0 = np.clip(b0, 0, nb - 1); b1 = np.clip(b1, 0, nb - 1)
    cnt = b1 - b0 + 1
    idx = np.repeat(np.arange(len(s)), cnt)
    off = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    blk = b0[idx] + off
    ps = np.maximum(s[idx], t0 + blk * tau)
    pe = np.minimum(e[idx], t0 + (blk + 1) * tau)
    m = pe > ps
    return acct[idx][m], blk[m], ps[m], pe[m]


def occupancy(acct, s, e, t0, t1, tau=None, n_accounts=None):
    """Occupancy moments over [t0, t1].
    tau=None -> full-window variances; tau=float -> mean over blocks of within-block variances
    (the 'high-pass' statistic). Returns dict with R, V_N, V_i (per account), mean_N, n_blocks."""
    acct = np.asarray(acct, np.int64)
    A = int(n_accounts if n_accounts is not None else (acct.max() + 1 if len(acct) else 0))
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9))
    if nb < 1:
        raise ValueError("window shorter than one block")
    t1 = t0 + nb * tau
    a, s, e = _clip(acct, s, e, t0, t1)
    a, b, s, e = _split_blocks(a, s, e, t0, tau, nb)
    # aggregate N, per block
    I1, I2 = _group_integrals(b, s, e, nb)
    VN_b = I2 / tau - (I1 / tau) ** 2
    # per account per block (compress group ids)
    key = a * nb + b
    uk, inv = np.unique(key, return_inverse=True)
    J1, J2 = _group_integrals(inv, s, e, len(uk))
    Vg = J2 / tau - (J1 / tau) ** 2
    V_i = np.bincount((uk // nb).astype(np.int64), weights=Vg, minlength=A) / nb
    V_N = VN_b.mean()
    sV = V_i.sum()
    return dict(R=V_N / sV if sV > 0 else np.nan, V_N=V_N, V_i=V_i, sumV=sV,
                mean_N=I1.sum() / (nb * tau), n_blocks=nb, tau=tau)


def R_full(acct, s, e, t0, t1):
    return occupancy(acct, s, e, t0, t1, None)["R"]


def R_hp(acct, s, e, t0, t1, tau=60.0):
    return occupancy(acct, s, e, t0, t1, tau)["R"]


def fano_khat(acct, s, e, t0, t1):
    """Fano factor of N and the implied effective number of hidden slots K = <N>^2/(<N>-Var N)."""
    o = occupancy(acct, s, e, t0, t1, None)
    F = o["V_N"] / o["mean_N"] if o["mean_N"] > 0 else np.nan
    p = 1 - F
    K = o["mean_N"] / p if p > 0 else np.inf
    return dict(F=F, p_hat=p, K_hat=K, mean_N=o["mean_N"], V_N=o["V_N"])


def baselines(acct, s, e, t0, t1):
    """Simple baselines: requests/hour, peak concurrency, Fano factor of N."""
    a, sc, ec = _clip(acct, s, e, t0, t1)
    n_req = int(((np.asarray(s) >= t0) & (np.asarray(s) < t1)).sum())
    t = np.concatenate([sc, ec]); d = np.concatenate([np.ones(len(sc)), -np.ones(len(ec))])
    o = np.lexsort((d, t)); peak = int(np.cumsum(d[o]).max()) if len(t) else 0
    f = fano_khat(acct, s, e, t0, t1)
    return dict(req_per_h=n_req / ((t1 - t0) / 3600), peak_N=peak, fano=f["F"])


# --------------------------------------------------------------------------------------------
# nulls
# --------------------------------------------------------------------------------------------
def circular_shift(acct, s, e, t0, t1, rng):
    """Shift each account's timeline by an independent uniform offset, wrapping around [t0, t1).
    Keeps each account's marginal behaviour and autocorrelation; destroys cross-account dependence."""
    acct = np.asarray(acct, np.int64); L = t1 - t0
    a, s, e = _clip(acct, s, e, t0, t1)
    A = int(acct.max() + 1) if len(acct) else 0
    off = rng.uniform(0, L, A)[a]
    s2 = t0 + np.mod(s - t0 + off, L); e2 = s2 + (e - s)
    wrap = e2 > t1
    a_out = np.concatenate([a, a[wrap]])
    s_out = np.concatenate([s2, np.full(wrap.sum(), t0)])
    e_out = np.concatenate([np.minimum(e2, t1), t0 + (e2[wrap] - t1)])
    return a_out, s_out, e_out


def surrogate_null(acct, s, e, t0, t1, tau, n_sur, rng):
    """Distribution of R (tau=None) or R_tau under independent circular shifts."""
    return np.array([occupancy(*circular_shift(acct, s, e, t0, t1, rng), t0, t1, tau)["R"] for _ in range(n_sur)])


# --------------------------------------------------------------------------------------------
# pool growth: correlation of each candidate's occupancy with a pool's aggregate occupancy
# --------------------------------------------------------------------------------------------
def _cum_integral(s, e, t0, t1):
    """Knots and running integral F(t) = int_t0^t N(u) du for the superposition of intervals."""
    t = np.concatenate([[t0], s, e, [t1]])
    d = np.concatenate([[0.0], np.ones(len(s)), -np.ones(len(e)), [0.0]])
    o = np.lexsort((d, t)); t, d = t[o], d[o]
    x = np.cumsum(d)
    F = np.concatenate([[0.0], np.cumsum(x[:-1] * np.diff(t))])
    return t, F


def corr_with_pool(pool, cand, t0, t1, tau=None, n_cand=None):
    """pool=(acct,s,e) of pool members; cand=(acct,s,e) of candidates (acct = candidate index 0..C-1).
    Returns per-candidate correlation of X_j with N_pool (within-block if tau is set). Linear in traffic."""
    pa, ps, pe = _clip(*pool, t0, t1)
    ca, cs, ce = _clip(*cand, t0, t1)
    C = int(n_cand if n_cand is not None else np.asarray(cand[0]).max() + 1)
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9)); t1 = t0 + nb * tau
    # pool per-block mean and within-block variance
    _, pb, pps, ppe = _split_blocks(pa, np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, tau, nb)
    I1, I2 = _group_integrals(pb, pps, ppe, nb)
    Nbar = I1 / tau; VN = (I2 / tau - Nbar ** 2).mean()
    kn, F = _cum_integral(np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, t1)
    # candidate pieces
    ca2, cb, cs2, ce2 = _split_blocks(ca, np.clip(cs, t0, t1), np.clip(ce, t0, t1), t0, tau, nb)
    XN = np.interp(ce2, kn, F) - np.interp(cs2, kn, F)       # int over piece of N_pool
    key = ca2 * nb + cb
    uk, inv = np.unique(key, return_inverse=True)
    A = np.bincount(inv, weights=XN)                          # int X_j N over block
    B = np.bincount(inv, weights=ce2 - cs2)                   # int X_j over block
    jj = uk // nb; bb = uk % nb
    cov = np.bincount(jj, weights=A / tau - (B / tau) * Nbar[bb], minlength=C) / nb
    Vj = occupancy(ca, cs, ce, t0, t1, None if nb == 1 else tau, n_accounts=C)["V_i"]
    with np.errstate(invalid="ignore", divide="ignore"):
        return cov / np.sqrt(Vj * VN)


# --------------------------------------------------------------------------------------------
# generators (simulation)
# --------------------------------------------------------------------------------------------
DUR_MED, DUR_SIG = 8.0, 0.6       # streamed response duration, log-normal
GAP_MED, GAP_SIG = 0.4, 0.5       # client processing gap, log-normal
E_DUR = DUR_MED * np.exp(DUR_SIG ** 2 / 2)
E_GAP = GAP_MED * np.exp(GAP_SIG ** 2 / 2)


def duty(J):
    return E_DUR / (E_DUR + E_GAP + J / 2.0)


def theory_R(p, M):
    return (1 - p) / (1 - p / M)


def _pack(rows):
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0), np.zeros(0)
    a, s, e = map(np.asarray, zip(*rows))
    return a.astype(np.int64), s.astype(float), e.astype(float)


def benign_humans(rng, n_accounts, T, session_rate=1 / 7200, mean_req=6, think_med=25.0, think_sig=0.9,
                  diurnal=0.0, offset=0):
    """Human-like accounts: Poisson session starts, geometric #requests, log-normal think time.
    diurnal in [0,1): amplitude of a 24 h sinusoidal rate modulation (common to all accounts)."""
    rows = []
    for i in range(n_accounts):
        t = rng.exponential(1 / session_rate)
        while t < T:
            if diurnal > 0 and rng.random() > (1 + diurnal * np.sin(2 * np.pi * t / 86400)) / (1 + diurnal):
                t += rng.exponential(1 / session_rate); continue
            tt = t
            for _ in range(rng.geometric(1 / mean_req)):
                if tt > T: break
                d = rng.lognormal(np.log(DUR_MED), DUR_SIG); rows.append((offset + i, tt, tt + d))
                tt += d + rng.lognormal(np.log(think_med), think_sig)
            t = tt + rng.exponential(1 / session_rate)
    return _pack(rows)


def closed_loop_pool(rng, T, K, M, J=0.0, assign="random", stint=None, K_bar=None, epoch=None, offset=0):
    """K closed-loop slots over M accounts. Each slot: send, wait for stream end, gap, uniform(0,J) delay.
    assign: 'random' (per request), 'pinned' (slot w -> account w % M), 'stint' (hold an account for `stint` s).
    K_bar/epoch: if set, the number of active slots is redrawn ~ Poisson(K_bar) (capped at K) every `epoch` s."""
    rows = []
    Kact = None
    if K_bar is not None:
        Kact = np.minimum(rng.poisson(K_bar, int(T // epoch) + 2), K)
    held = {}
    for w in range(K):
        t = rng.uniform(0, 5)
        while t < T:
            if Kact is not None:
                ei = int(t // epoch)
                if w >= Kact[ei]:
                    t = (ei + 1) * epoch + rng.uniform(0, 1); continue
            if assign == "pinned":
                a = w % M
            elif assign == "stint":
                k = (w, int(t // stint))
                if k not in held: held[k] = rng.integers(M)
                a = held[k]
            else:
                a = rng.integers(M)
            d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
            rows.append((offset + a, t, t + d))
            t += d + rng.lognormal(np.log(GAP_MED), GAP_SIG) + (rng.uniform(0, J) if J > 0 else 0.0)
    return _pack(rows)


def open_loop_relay(rng, T, M, lam, policy="random", offset=0):
    """Open-loop Poisson arrivals (rate lam/s) dispatched over M accounts: random | rr | least."""
    import heapq
    rows = []; inflight = np.zeros(M, int); heap = []; t = 0.0; rr = 0
    while True:
        t += rng.exponential(1 / lam)
        if t > T: break
        while heap and heap[0][0] <= t:
            _, a = heapq.heappop(heap); inflight[a] -= 1
        if policy == "random": a = int(rng.integers(M))
        elif policy == "rr": a = rr % M; rr += 1
        else:
            m = inflight.min(); a = int(rng.choice(np.flatnonzero(inflight == m)))
        d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
        rows.append((offset + a, t, t + d)); inflight[a] += 1; heapq.heappush(heap, (t + d, a))
    return _pack(rows)


def concat(*logs):
    return tuple(np.concatenate(x) for x in zip(*logs))


def relabel(acct):
    """Map arbitrary account ids to 0..A-1."""
    u, inv = np.unique(acct, return_inverse=True)
    return inv.astype(np.int64), u


In [ ]:
%%writefile fplib.py
"""fplib — dispatcher fingerprinting from per-key request logs (start, end) only.

Signals (all computed without labels):
  ring     zero-lag high-pass occupancy correlation arranged on a recovered circle (shared round-robin counter)
  conc     concentration of each key's short-lag end->start successors (per-worker round-robin counters)
  coll     same-key overlap relative to uniform spreading of concurrent requests (least-busy / round-robin avoid collisions)
  R(tau)   occupancy variance ratio at several timescales (closed-loop turn-taking, sticky/episodic key holding)
  K_little worker count from the interactive response-time law, think time read off the successor-lag excess
"""
import numpy as np

__version__ = "fp-0.6"


# ---------------------------------------------------------------- occupancy on a grid
def occupancy_matrix(a, s, e, M, t0, t1, dt=0.05):
    n = int((t1 - t0) / dt)
    i0 = np.clip(np.floor((s - t0) / dt).astype(np.int64), 0, n)
    i1 = np.clip(np.floor((e - t0) / dt).astype(np.int64), 0, n)
    D = np.zeros((M, n + 1), np.float32)
    np.add.at(D, (a, i0), 1.0); np.add.at(D, (a, i1), -1.0)
    return np.cumsum(D, 1)[:, :n]


def highpass(X, block):
    M, n = X.shape; nb = n // block
    Y = X[:, : nb * block].reshape(M, nb, block)
    return (Y - Y.mean(2, keepdims=True)).reshape(M, nb * block)


def corr_matrix(X):
    C = np.cov(X.astype(np.float64)); d = np.sqrt(np.clip(np.diag(C), 0, None))
    with np.errstate(invalid="ignore", divide="ignore"):
        R = C / np.outer(d, d)
    R[~np.isfinite(R)] = 0.0
    return R


# ---------------------------------------------------------------- circular seriation
def circular_seriation(C):
    """Order keys around a circle from a similarity matrix (angle in the plane of two leading eigenvectors)."""
    S = np.clip(C, 0, None).copy(); np.fill_diagonal(S, 0)
    w, V = np.linalg.eigh(S)
    best = None
    for i, j in [(1, 2), (2, 3), (1, 3)]:
        x, y = V[:, -i], V[:, -j]
        r = np.hypot(x, y)
        score = r.std() / (r.mean() + 1e-12)          # points of a ring embed at constant radius
        if best is None or score < best[0]:
            best = (score, np.arctan2(y, x))
    return np.argsort(best[1])


def ring_score(C, order):
    """mean correlation between circular neighbours in the recovered order minus mean off-diagonal correlation"""
    M = len(order)
    nb = np.mean([C[order[i], order[(i + 1) % M]] for i in range(M)])
    off = C[~np.eye(M, dtype=bool)].mean()
    return float(nb - off)


def neighbour_accuracy(order, true_pos):
    """fraction of recovered neighbours that are true ring neighbours; true_pos[k] = key k's true ring position"""
    M = len(order); p = np.asarray(true_pos)[order]
    return float(np.mean([((p[i] - p[(i + 1) % M]) % M) in (1, M - 1) for i in range(M)]))


# ---------------------------------------------------------------- end -> start successors
def handoff_counts(a, s, e, M, lag_lo=0.0, lag_hi=1.0):
    """H[i, j] = number of (end on key i, start on key j within (lag_lo, lag_hi] s after that end)"""
    o = np.argsort(s, kind="stable"); ss, sa = s[o], a[o]
    lo = np.searchsorted(ss, e + lag_lo, "right"); hi = np.searchsorted(ss, e + lag_hi, "right")
    cnt = hi - lo
    H = np.zeros((M, M))
    if cnt.sum() == 0:
        return H
    src = np.repeat(a, cnt)
    idx = np.repeat(lo - np.r_[0, np.cumsum(cnt)[:-1]], cnt) + np.arange(cnt.sum())
    np.add.at(H, (src, sa[idx]), 1)
    return H


def handoff_excess(a, s, e, M, t0, t1, lag_hi=1.0):
    """observed minus expected successor counts (expected: key j's starts spread uniformly over time)"""
    H = handoff_counts(a, s, e, M, 0.0, lag_hi)
    ends = np.bincount(a, minlength=M).astype(float); rate = ends / (t1 - t0)
    Ex = np.outer(ends, rate) * lag_hi
    return H, Ex


def successor_concentration(a, s, e, M, t0, t1, lag_hi=1.0, min_rows=5):
    H, Ex = handoff_excess(a, s, e, M, t0, t1, lag_hi)
    X = np.clip(H - Ex, 0, None); np.fill_diagonal(X, 0)
    rows = X.sum(1); ok = rows >= min_rows
    if ok.sum() < 3:
        return float("nan"), X
    return float(np.mean(X[ok].max(1) / rows[ok])), X


def ring_offsets(X, order):
    """distribution of successor offsets measured along the recovered circle (direction chosen by mass)"""
    M = len(order); pos = np.empty(M, int); pos[order] = np.arange(M)
    off = np.zeros(M)
    I, J = np.nonzero(X)
    np.add.at(off, (pos[J] - pos[I]) % M, X[I, J])
    h = M // 2
    if off[h + 1:].sum() > off[1:h].sum():
        off = np.r_[off[0], off[1:][::-1]]
    return off


def K_ring(X, order):
    off = ring_offsets(X, order); M = len(order); h = M // 2
    w = off[1:h]
    return float((np.arange(1, h) * w).sum() / w.sum()) if w.sum() > 0 else float("nan")


# ---------------------------------------------------------------- collisions
def collision_ratio(a, s, e, M, t0, t1):
    """time-integrated same-key concurrent pairs / expected if concurrent requests were spread uniformly over M keys"""
    ev = np.r_[s, e]; dlt = np.r_[np.ones(len(s), int), -np.ones(len(e), int)]; kk = np.r_[a, a]
    o = np.lexsort((dlt, ev)); ev, dlt, kk = ev[o], dlt[o], kk[o]
    X = np.zeros(M, int); N = 0; pairs = 0; obs = exp_ = 0.0; last = t0
    for t, d, k in zip(ev.tolist(), dlt.tolist(), kk.tolist()):
        span = min(t, t1) - max(last, t0)
        if span > 0:
            obs += pairs * span; exp_ += N * (N - 1) / 2.0 / M * span
        last = t
        if d > 0: pairs += X[k]
        else: pairs -= X[k] - 1
        X[k] += d; N += d
    return float(obs / exp_) if exp_ > 0 else float("nan")


# ---------------------------------------------------------------- surrogates
def shift_keys(a, s, e, t0, t1, rng, M):
    """independent circular time shift per key (keeps each key's own pattern, destroys cross-key timing)"""
    L = t1 - t0; off = rng.uniform(0, L, M)[a]
    s2 = t0 + np.mod(s - t0 + off, L)
    return a, s2, s2 + (e - s)


# ---------------------------------------------------------------- worker count from the interactive response-time law
LAG_EDGES = np.r_[0, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 12, 16]


def excess_lag_hist(s, e, t0, t1, edges=LAG_EDGES):
    """pool-level histogram of (end -> any later start in the pool) lags minus its expectation if starts were uniform in time"""
    o = np.sort(s); lam = len(s) / (t1 - t0)
    return np.array([(np.searchsorted(o, e + hi, "right") - np.searchsorted(o, e + lo, "right")).sum() - len(e) * lam * (hi - lo)
                     for lo, hi in zip(edges[:-1], edges[1:])])


def K_little(s, e, t0, t1, edges=LAG_EDGES):
    """K = X * (E[D] + E[G]) (Denning-Buzen interactive response time law, N = X (R + Z)) with throughput X, mean response
    E[D] and the client gap ('think time') E[G] read off the excess successor-lag histogram. Valid for always-busy closed loops."""
    h = np.clip(excess_lag_hist(s, e, t0, t1, edges), 0, None); mid = (edges[:-1] + edges[1:]) / 2
    EG = float((h * mid).sum() / h.sum()) if h.sum() > 0 else float("nan")
    lam = len(s) / (t1 - t0)
    return lam * (float(np.mean(e - s)) + EG), EG


# ---------------------------------------------------------------- full fingerprint of one pool
def best_lag_window(a, s, e, M, t0, t1, windows=(0.5, 1.0, 2.0, 4.0, 8.0)):
    """successor window with the largest cross-key excess z (sharpest view of the closed-loop hand-off)"""
    off = ~np.eye(M, dtype=bool); best = (windows[0], -np.inf)
    for w in windows:
        H, Ex = handoff_excess(a, s, e, M, t0, t1, w)
        z = (H[off].sum() - Ex[off].sum()) / np.sqrt(Ex[off].sum() + 1)
        if z > best[1]: best = (w, float(z))
    return best


def fingerprint(a, s, e, M, t0, t1, rng, n_sur=40, dt=0.05, hp_block_s=20.0, lag_hi=None, occupancy_fn=None):
    """a: keys 0..M-1 (requests overlapping [t0, t1)). Returns (features, recovered circular order)."""
    m = (e > t0) & (s < t1); a, s, e = a[m], s[m], e[m]
    out = dict(n_req=int(len(a)), n_keys=int(M))
    if lag_hi is None:
        lag_hi, out["handoff_z"] = best_lag_window(a, s, e, M, t0, t1)
    out["lag_win"] = lag_hi
    if occupancy_fn is not None:
        for tau in (None, 5.0, 20.0, 120.0):
            if tau is None or (t1 - t0) >= 2 * tau:
                out["R" if tau is None else f"R{int(tau)}"] = float(occupancy_fn(a, s, e, t0, t1, tau, n_accounts=M)["R"])
    out["coll"] = collision_ratio(a, s, e, M, t0, t1)
    out["K_little"], out["think_s"] = K_little(s, e, t0, t1)
    def ring_of(a_, s_, e_):
        C = corr_matrix(highpass(occupancy_matrix(a_, s_, e_, M, t0, t1, dt), int(hp_block_s / dt)))
        o = circular_seriation(C); return ring_score(C, o), o
    out["ring"], order = ring_of(a, s, e)
    out["conc"], X = successor_concentration(a, s, e, M, t0, t1, lag_hi)
    if n_sur == 0:
        return out, order
    sr, sc = [], []
    for _ in range(n_sur):
        aa, ss, ee = shift_keys(a, s, e, t0, t1, rng, M)
        sr.append(ring_of(aa, ss, ee)[0]); sc.append(successor_concentration(aa, ss, ee, M, t0, t1, lag_hi)[0])
    sr, sc = np.array(sr), np.array(sc)
    out["ring_p"] = float((1 + (sr >= out["ring"]).sum()) / (n_sur + 1))
    out["ring_z"] = float((out["ring"] - sr.mean()) / (sr.std() + 1e-12))
    sc = sc[np.isfinite(sc)]
    ok = len(sc) > 2 and np.isfinite(out["conc"])
    out["conc_z"] = float((out["conc"] - sc.mean()) / (sc.std() + 1e-12)) if ok else float("nan")
    out["conc_p"] = float((1 + (sc >= out["conc"]).sum()) / (len(sc) + 1)) if ok else float("nan")
    return out, order


# ---------------------------------------------------------------- pre-registered classifier
CLASSES = ["none", "shared round-robin (closed loop)", "shared round-robin (open-loop relay)", "per-worker round-robin",
           "random / shuffle", "least-busy", "sticky / episodic", "bunching"]


def classify(f, zR, z_thr=3.0, ring_z_thr=6.0, conc_z_thr=6.0):
    """f: fingerprint dict; zR: dict of z-scores of R-type statistics against the human null (positive = turn-taking).
    Thresholds are fixed in advance from simulation (fp/explore*.py), not tuned on the testbed."""
    z_full, z_short = zR.get("R", 0.0), zR.get("R5", 0.0)
    turn = max(z_full, zR.get("R20", 0.0)) > z_thr
    if f["ring_z"] > ring_z_thr:
        return CLASSES[1] if turn else CLASSES[2]
    if np.isfinite(f.get("conc_z", np.nan)) and f["conc_z"] > conc_z_thr:
        return CLASSES[3]
    if turn:
        r_full, r5 = f.get("R", 1.0), f.get("R5", 1.0)
        if (1 - r5) < 0.5 * (1 - r_full):
            return CLASSES[6]
        return CLASSES[5] if f["coll"] < 0.5 else CLASSES[4]
    if min(z_full, zR.get("R20", 0.0)) < -z_thr:
        return CLASSES[7]
    return CLASSES[0]


# ---------------------------------------------------------------- site-wide blind discovery
def site_handoff_graph(a, s, e, N, t0, t1, lag_hi=1.0, q=1e-3, min_count=4):
    """Symmetric key-pair successor excess with exact Poisson tails, Benjamini-Hochberg over all pairs.
    Returns component labels (-1 = not in any component) and the edge list."""
    from scipy.stats import poisson
    H, Ex = handoff_excess(a, s, e, N, t0, t1, lag_hi)
    np.fill_diagonal(H, 0); np.fill_diagonal(Ex, 0)
    Hs, Es = H + H.T, Ex + Ex.T
    iu = np.triu_indices(N, 1)
    obs, exp_ = Hs[iu], Es[iu]
    p = poisson.sf(obs - 1, np.maximum(exp_, 1e-12))
    p[obs < min_count] = 1.0
    m = len(p); o = np.argsort(p); thr = q * np.arange(1, m + 1) / m
    passed = np.zeros(m, bool)
    k = np.nonzero(p[o] <= thr)[0]
    if len(k): passed[o[: k.max() + 1]] = True
    I, J = iu[0][passed], iu[1][passed]
    parent = np.arange(N)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in zip(I, J):
        ri, rj = find(i), find(j)
        if ri != rj: parent[ri] = rj
    roots = np.array([find(i) for i in range(N)])
    deg = np.bincount(np.r_[I, J], minlength=N)
    lab = np.where(deg > 0, roots, -1)
    return lab, list(zip(I.tolist(), J.tolist()))


# ================================================================ v0.4 additions (fixed after NB6, frozen for NB7)
def sideband_expect(starts_j, ends_i, w=1.0, B=60.0):
    """observed and local-rate expected count of key-j starts within (e, e+w] after key-i ends.
    Expectation uses j's own starts in [e-B, e+B] outside (e-w, e+w] (an off-source sideband), so bursty sessions cancel."""
    obs = (np.searchsorted(starts_j, ends_i + w, "right") - np.searchsorted(starts_j, ends_i, "right")).sum()
    side = (np.searchsorted(starts_j, ends_i + B, "right") - np.searchsorted(starts_j, ends_i - B, "left")).sum() - \
           (np.searchsorted(starts_j, ends_i + w, "right") - np.searchsorted(starts_j, ends_i - w, "left")).sum()
    return float(obs), float(side) * w / (2 * B - 2 * w)


def site_handoff_graph_sideband(a, s, e, N, t0, t1, lag_hi=1.0, q=1e-3, min_count=4, B=60.0, alpha=1e-3):
    """Stage 1: Poisson screen with Benjamini-Hochberg (site_handoff_graph). Stage 2: keep an edge only if it also beats the
    local-rate sideband null (Bonferroni over screened edges). Returns component labels (-1 = none) and kept edges."""
    from scipy.stats import poisson
    _, edges = site_handoff_graph(a, s, e, N, t0, t1, lag_hi, q, min_count)
    st = {k: np.sort(s[a == k]) for k in set(np.ravel(edges).tolist())} if edges else {}
    en = {k: e[a == k] for k in st}
    kept = []
    for i, j in edges:
        o1, e1 = sideband_expect(st[j], en[i], lag_hi, B); o2, e2 = sideband_expect(st[i], en[j], lag_hi, B)
        if poisson.sf(o1 + o2 - 1, max(e1 + e2, 1e-9)) < alpha / len(edges):
            kept.append((i, j))
    parent = np.arange(N)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i, j in kept:
        parent[find(i)] = find(j)
    deg = np.bincount(np.ravel(kept).astype(int), minlength=N) if kept else np.zeros(N, int)
    return np.where(deg > 0, [find(i) for i in range(N)], -1), kept


def hollow_corr(a, s, e, N, t0, t1, dt=0.1, hp_block_s=20.0):
    """zero-lag high-pass occupancy correlation over all keys, site-wide common mode removed, diagonal set to 0"""
    X = highpass(occupancy_matrix(a, s, e, N, t0, t1, dt).astype(np.float64), int(hp_block_s / dt))
    sd = X.std(1); ok = sd > 0
    Z = np.zeros_like(X); Z[ok] = (X[ok] - X[ok].mean(1, keepdims=True)) / sd[ok, None]
    g = Z[ok].mean(0); g /= np.linalg.norm(g) + 1e-12
    Z[ok] -= np.outer(Z[ok] @ g, g)
    C = Z @ Z.T / Z.shape[1]; np.fill_diagonal(C, 0.0)
    return C


def spectral_rings(a, s, e, N, t0, t1, rng, n_null=3, edge_factor=1.05, pair_tol=0.9, load_k=2.5, hp_block_s=20.0):
    """Rings as outlier eigen-pairs of the hollowed correlation matrix above a time-shifted null bulk.
    Returns labels (-1 = none), list of eigen-groups with eigenvalues, the bulk edge, and per-group circular order."""
    C = hollow_corr(a, s, e, N, t0, t1, hp_block_s=hp_block_s)
    w, V = np.linalg.eigh(C)
    tops = []
    for _ in range(n_null):
        C_null = hollow_corr(*shift_keys(a, s, e, t0, t1, rng, N), N, t0, t1, hp_block_s=hp_block_s)
        tops.append(np.linalg.eigvalsh(C_null)[-1])
    edge = edge_factor * max(tops)
    out = np.nonzero(w > edge)[0][::-1]
    groups, i = [], 0
    while i < len(out):
        if i + 1 < len(out) and w[out[i + 1]] > pair_tol * w[out[i]]:
            groups.append([int(out[i]), int(out[i + 1])]); i += 2
        else:
            groups.append([int(out[i])]); i += 1
    lab = -np.ones(N, int); orders = []
    if groups:
        L = np.stack([np.sqrt((V[:, g] ** 2).sum(1)) for g in groups], 1)
        # v0.5: one ring can give several outlier pairs (its Fourier harmonics), so keys are not assigned to eigen-groups.
        # Select keys that load on any outlier group, then split them into rings by strong pairwise correlation
        # (neighbours on a ring correlate at ~0.5; unrelated keys at ~0 +- sigma), threshold max(0.2, 6 sigma_null).
        import scipy.sparse as _sp, scipy.sparse.csgraph as _cg
        sel = np.nonzero(L.max(1) > load_k / np.sqrt(N))[0]
        iu = np.triu_indices(N, 1)
        thr = max(0.2, 6.0 * float(np.sqrt(np.mean(C_null[iu] ** 2))))
        if len(sel):
            Cs = C[np.ix_(sel, sel)]
            ncomp, cl = _cg.connected_components(_sp.csr_matrix(Cs > thr), directed=False)
            nxt = 0
            for c in range(ncomp):
                mem = sel[cl == c]
                if len(mem) >= 5:
                    lab[mem] = nxt; nxt += 1
                    orders.append(mem[circular_seriation(C[np.ix_(mem, mem)])])
    return lab, [[float(w[j]) for j in g] for g in groups], float(edge), orders


# ---------------------------------------------------------------- active timing watermark (randomisation test)
def watermark_echo(a, s, c, b, src_keys, dst_keys, w=1.0, n_perm=2000, rng=None, same_key=False):
    """Provider delayed the end of each response by b*delta (b = secret fair coin). For every request on a source key,
    take the mean lag, from its true completion c, of starts on destination keys within [c, c+w) (other keys only, or the
    same key if same_key). Statistic T = mean lag | b=1 minus mean lag | b=0. Exact p-value by re-drawing the coins."""
    rng = rng or np.random.default_rng(0)
    src = np.isin(a, src_keys); dst = np.isin(a, dst_keys)
    o = np.argsort(s[dst]); ss, sa = s[dst][o], a[dst][o]
    idx = np.nonzero(src)[0]
    i0 = np.searchsorted(ss, c[idx], "left"); i1 = np.searchsorted(ss, c[idx] + w, "left")
    L = np.full(len(idx), np.nan)
    for n, (k, x, y) in enumerate(zip(idx, i0, i1)):
        if y > x:
            m = (sa[x:y] == a[k]) if same_key else (sa[x:y] != a[k])
            if m.any():
                L[n] = (ss[x:y][m] - c[k]).mean()
    ok = np.isfinite(L); L, bb = L[ok], b[idx][ok]
    if (bb == 1).sum() < 10 or (bb == 0).sum() < 10:
        return dict(n=int(ok.sum()), shift_ms=float("nan"), z=float("nan"), p=float("nan"))
    T = L[bb == 1].mean() - L[bb == 0].mean()
    perm = np.empty(n_perm)
    for r in range(n_perm):
        q = rng.permutation(bb); perm[r] = L[q == 1].mean() - L[q == 0].mean()
    return dict(n=int(ok.sum()), shift_ms=float(T * 1000), z=float((T - perm.mean()) / (perm.std() + 1e-12)),
                p=float((1 + (perm >= T).sum()) / (n_perm + 1)))


# ================================================================ v0.6 (defined after the NB8 dry run, before any NB8 data on vLLM)
# A ring whose period T_c = M / lambda exceeds the high-pass block is removed with the slow trend. Long answers make T_c long,
# so ring statistics are computed at several block lengths; v0.5 (20 s only) stays as the frozen reference.
SCALES = (20.0, 60.0, 180.0)


def ring_multiscale(a, s, e, M, t0, t1, rng, n_sur=40, scales=SCALES):
    """max over scales of the ring z (surrogates per scale); returns (z_max, best scale, order at that scale)"""
    best = (-np.inf, None, None)
    for sc in scales:
        if (t1 - t0) < 4 * sc: continue
        f, order = fingerprint(a, s, e, M, t0, t1, rng, n_sur=n_sur, hp_block_s=sc)
        if f["ring_z"] > best[0]: best = (f["ring_z"], sc, order)
    return best


def spectral_rings_multiscale(a, s, e, N, t0, t1, rng, scales=SCALES, **kw):
    """union of spectral_rings over block lengths; rings found at several scales are merged when they share keys"""
    labs = []
    for sc in scales:
        if (t1 - t0) < 4 * sc: continue
        lab, groups, edge, orders = spectral_rings(a, s, e, N, t0, t1, rng, hp_block_s=sc, **kw)
        labs.append(lab)
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x: x = parent[x]
        return x
    for li, lab in enumerate(labs):
        for c in np.unique(lab[lab >= 0]):
            mem = np.nonzero(lab == c)[0]
            for k in mem: parent[find(("k", int(k)))] = find(("r", li, int(c)))
    out = -np.ones(N, int); roots = {}
    for k in range(N):
        if ("k", k) in parent:
            r = find(("k", k)); out[k] = roots.setdefault(r, len(roots))
    return out


In [ ]:
%%writefile nb8an.py
"""NB8 analysis: one function per phase. Labels are used only to say which keys form a pool and to score answers."""
import json, glob, os, time
import numpy as np, pandas as pd
import occlib, fplib
from occlib import occupancy, fano_khat

RKEYS = ["R", "R5", "R20", "R120"]


def load(run_dir, cfg):
    labels = {}
    for f in glob.glob(os.path.join(run_dir, "labels_*.json")):
        labels.update(json.load(open(f)))
    log = pd.read_csv(os.path.join(run_dir, "gateway_log.csv"), dtype={"rid": str})
    log["grp"] = log.key.map(lambda k: labels.get(k, {}).get("pop"))
    log.loc[log.grp.isin(["hum_a", "hum_b"]), "grp"] = "hum"
    log["pos"] = log.key.map(lambda k: labels.get(k, {}).get("pos"))
    okm = (log.status == 200) & log.grp.notna()
    t0 = cfg["t_start"] + cfg["warmup_s"]; t1 = min(float(log.t_end.max()), cfg["t_start"] + cfg["run_s"]) - 30
    d = log[okm & (log.t_end > t0) & (log.t_arrival < t1)].copy()
    d["dur"] = d.t_end - d.t_arrival
    cl = []
    for f in glob.glob(os.path.join(run_dir, "clientlog_*.csv")):
        x = pd.read_csv(f, dtype={"rid": str}); x["src"] = os.path.basename(f)[len("clientlog_"):-4]; cl.append(x)
    cl = pd.concat(cl, ignore_index=True) if cl else pd.DataFrame()
    return d, labels, cl, t0, t1, dict(n=len(log), ok_frac=round(float(okm.mean()), 4), by_pop=log[okm].grp.value_counts().to_dict())


def analyze(run_dir, cfg, rng, phase):
    t_an = time.time()
    d, labels, cl, t0, t1, logsum = load(run_dir, cfg)
    out = {"log": logsum, "window_s": round(t1 - t0), "durations_s": d.groupby("grp").dur.median().round(2).to_dict()}
    keys_by = {g: sorted(v) for g, v in d.groupby("grp").key.unique().items()}

    def pool_arrays(ks_):
        ks_ = sorted(ks_); idx = {k: i for i, k in enumerate(ks_)}; m = d.key.isin(idx)
        return d.key[m].map(idx).values.astype(np.int64), d.t_arrival.values[m], d.t_end.values[m], ks_

    # ---- human null for R-type statistics
    hum = keys_by.get("hum", []); bat = keys_by.get("bat", [])
    null = []
    for _ in range(200):
        a, s, e, ks = pool_arrays(list(rng.choice(hum, min(40, len(hum)), replace=False)))
        st = {"R": float(occupancy(a, s, e, t0, t1, None, n_accounts=len(ks))["R"])}
        for tau in (5.0, 20.0, 120.0):
            st[f"R{int(tau)}"] = float(occupancy(a, s, e, t0, t1, tau, n_accounts=len(ks))["R"])
        null.append(st)
    NULL = {k: np.array([x[k] for x in null]) for k in RKEYS}
    out["null_humans_40"] = {k: dict(mean=round(float(v.mean()), 4), sd=round(float(v.std()), 4),
                                     frac_abs_z_gt3=round(float((np.abs(v - v.mean()) / v.std() > 3).mean()), 4)) for k, v in NULL.items()}
    zR = lambda st: {k: float((NULL[k].mean() - st[k]) / NULL[k].std()) for k in RKEYS if k in st}

    # ---- fingerprint + frozen classifier
    exp = cfg["expected"][phase]
    pools = {p: keys_by[p] for p in exp if p in keys_by}
    pools["hum_random40"] = list(rng.choice(hum, min(40, len(hum)), replace=False))
    if bat:
        pools["benign_36hum_4bat"] = list(rng.choice(hum, min(36, len(hum)), replace=False)) + list(bat)
    rows = {}
    for name, ks_ in pools.items():
        a, s, e, ks = pool_arrays(ks_); M = len(ks)
        f, order = fplib.fingerprint(a, s, e, M, t0, t1, rng, n_sur=40, occupancy_fn=occupancy)
        z = zR(f); cls = fplib.classify(f, z); fk = fano_khat(a, s, e, t0, t1)
        rz6, sc6, order6 = fplib.ring_multiscale(a, s, e, M, t0, t1, rng, n_sur=40)
        f6 = dict(f, ring_z=max(f["ring_z"], rz6)); cls6 = fplib.classify(f6, z)
        if rz6 > f["ring_z"] and order6 is not None: order = order6
        true_pos = np.array([labels[k]["pos"] for k in ks]) if all(k in labels for k in ks) else np.arange(M)
        P = cfg["pops"].get(name, {})
        lam_ = f["n_req"] / (t1 - t0)
        rows[name] = dict(expected=exp.get(name, "none"), predicted=cls, correct=cls in exp.get(name, "none").split(" | "),
                          predicted_v06=cls6, correct_v06=cls6 in exp.get(name, "none").split(" | "),
                          ring_z_multiscale=round(float(rz6), 1) if np.isfinite(rz6) else None, ring_scale_s=sc6,
                          ring_period_s=round(M / lam_, 1) if lam_ > 0 else None,
                          n_req=f["n_req"], req_per_s=round(f["n_req"] / (t1 - t0), 3),
                          req_per_s_per_worker=round(f["n_req"] / (t1 - t0) / P["K"], 3) if P.get("K") else None,
                          R=round(f["R"], 3), R5=round(f["R5"], 3), R20=round(f["R20"], 3), R120=round(f["R120"], 3),
                          z_R=round(z["R"], 1), z_R20=round(z["R20"], 1), coll=round(f["coll"], 2), ring_z=round(f["ring_z"], 1),
                          conc_z=round(f["conc_z"], 1) if np.isfinite(f["conc_z"]) else None,
                          ring_acc=round(fplib.neighbour_accuracy(order, np.argsort(np.argsort(true_pos))), 2),
                          K_true=P.get("K") if P.get("kind") not in ("open_loop",) else None,
                          K_fano=round(float(fk["K_hat"]), 1) if np.isfinite(fk["K_hat"]) else None,
                          K_little=round(f["K_little"], 1) if np.isfinite(f["K_little"]) else None)
        print(f"[{phase}] {name:20s} v0.5 -> {cls:38s} {'OK' if rows[name]['correct'] else 'WRONG':5s} | v0.6 -> {cls6:38s} {'OK' if rows[name]['correct_v06'] else 'WRONG'}", flush=True)
    out["pools"] = rows

    # ---- site-wide discovery (frozen fplib v0.5)
    allk = sorted(d.key.unique()); idx = {k: i for i, k in enumerate(allk)}
    A = d.key.map(idx).values.astype(np.int64); S_, E_ = d.t_arrival.values, d.t_end.values; NK = len(allk)
    grp_of = np.array([labels[k]["pop"] if labels[k]["pop"] not in ("hum_a", "hum_b") else "hum" for k in allk])
    pos_of = np.array([labels[k]["pos"] for k in allk])

    def recovery(lab):
        res = {}
        for p in pools:
            if p.startswith(("hum", "benign")): continue
            tot = int((grp_of == p).sum()); best, pur = 0, None
            for c in np.unique(lab[lab >= 0]):
                n = int(((lab == c) & (grp_of == p)).sum())
                if n > best: best, pur = n, round(n / int((lab == c).sum()), 2)
            res[p] = dict(recall=round(best / tot, 2) if tot else None, purity=pur)
        return res
    lab_g, kept = fplib.site_handoff_graph_sideband(A, S_, E_, NK, t0, t1)
    lab_s, groups, bulk_edge, orders = fplib.spectral_rings(A, S_, E_, NK, t0, t1, rng)
    lab_s6 = fplib.spectral_rings_multiscale(A, S_, E_, NK, t0, t1, rng)
    ring_orders = {}
    for o in orders:
        if len(o) >= 5:
            g = pd.Series(grp_of[o]).mode()[0]
            ring_orders[g] = round(fplib.neighbour_accuracy(np.arange(len(o)), np.argsort(np.argsort(pos_of[o]))), 2)
    out["site"] = dict(sideband_graph=dict(edges=len(kept), humans=int(((lab_g >= 0) & (grp_of == "hum")).sum()), recovery=recovery(lab_g)),
                       spectral=dict(bulk_edge=round(bulk_edge, 2), eigen_groups=[[round(x, 2) for x in g] for g in groups],
                                     humans=int(((lab_s >= 0) & (grp_of == "hum")).sum()), recovery=recovery(lab_s), ring_order_acc=ring_orders),
                       spectral_v06=dict(humans=int(((lab_s6 >= 0) & (grp_of == "hum")).sum()), recovery=recovery(lab_s6)))

    # ---- watermark (exact randomisation tests)
    dw = d.copy(); dw["c"] = dw.t_complete; dw["b"] = dw.wm_bit.astype(int)
    out["watermark_cost_ms"] = dict(mean_added=round(float((dw.t_end - dw.c).mean() * 1000), 1),
                                    p50_with=round(float(np.median(dw.t_end - dw.t_arrival)), 3), p50_without=round(float(np.median(dw.c - dw.t_arrival)), 3),
                                    p99_with=round(float(np.quantile(dw.t_end - dw.t_arrival, .99)), 3), p99_without=round(float(np.quantile(dw.c - dw.t_arrival, .99)), 3))

    def wm(src, dst, w=1.0, same=False, n_perm=2000):
        kk = {k: i for i, k in enumerate(sorted(set(src) | set(dst)))}
        m = dw.key.isin(kk).values; a = dw.key[m].map(kk).values
        r = fplib.watermark_echo(a, dw.t_arrival.values[m], dw.c.values[m], dw.b.values[m], [kk[k] for k in src], [kk[k] for k in dst],
                                 w=w, n_perm=n_perm, rng=rng, same_key=same)
        return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}
    wmres = {}
    for p in [x for x in exp if x in keys_by]:
        ks = keys_by[p]; half = list(rng.permutation(ks)); hA, hB = half[: len(ks) // 2], half[len(ks) // 2:]
        wmres[p] = dict(pool_w1=wm(ks, ks, 1.0), pool_w8=wm(ks, ks, 8.0), halves_w1=wm(hA, hB, 1.0), same_key_w1=wm(ks, ks, 1.0, same=True))
    hp = []
    for _ in range(50):
        ks = list(rng.choice(hum, min(40, len(hum)), replace=False)); hp.append(wm(ks, ks, 1.0, n_perm=1000)["p"])
    hp = np.array([x for x in hp if np.isfinite(x)])
    out["watermark"] = wmres
    out["watermark_human_null"] = dict(n=int(len(hp)), n_p_below_0_01=int((hp < 0.01).sum()), n_p_below_0_05=int((hp < 0.05).sum()),
                                       median_p=round(float(np.median(hp)), 3) if len(hp) else None)
    # ---- can the client see the watermark? last inter-chunk gap by coin
    if len(cl) and "gap_last" in cl:
        j = dw.merge(cl[["rid", "gap_last", "gap_med"]], on="rid", how="inner")
        j = j[np.isfinite(j.gap_last)]
        if len(j):
            out["client_visibility_ms"] = dict(mode=cfg["wm_mode"][phase],
                                               last_gap_bit1=round(float(j.gap_last[j.b == 1].median() * 1000), 1),
                                               last_gap_bit0=round(float(j.gap_last[j.b == 0].median() * 1000), 1),
                                               median_gap=round(float(j.gap_med.median() * 1000), 1),
                                               auc_bit_from_last_gap=round(float(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(j.b, j.gap_last)), 3))
    out["analysis_min"] = round((time.time() - t_an) / 60, 1)
    return out


In [ ]:
%%writefile simserver.py
"""Simulated OpenAI-compatible streaming server (fallback when no GPU/vLLM).
Latency model: requests beyond `capacity` wait in a FIFO queue; per-token time grows with the number of active
sequences (batching interference): tpot = base_tpot * (1 + slowdown * active / capacity)."""
import asyncio, json, time, argparse, random
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=8000)
ap.add_argument("--capacity", type=int, default=128)
ap.add_argument("--base_tpot", type=float, default=0.02)
ap.add_argument("--slowdown", type=float, default=1.0)
ap.add_argument("--ttft", type=float, default=0.08)
args = ap.parse_args()
sem = asyncio.Semaphore(args.capacity)
active = 0

async def chat(request):
    global active
    body = await request.json()
    n = int(body.get("max_tokens") or 64)
    resp = web.StreamResponse(status=200, headers={"Content-Type": "text/event-stream"})
    await resp.prepare(request)
    async with sem:
        active += 1
        try:
            await asyncio.sleep(args.ttft * random.uniform(0.8, 1.2))
            for i in range(n):
                chunk = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                         "choices": [{"index": 0, "delta": {"content": f"w{random.randint(0, 9999)} "}, "finish_reason": None}]}
                await resp.write(f"data: {json.dumps(chunk)}\n\n".encode())
                await asyncio.sleep(args.base_tpot * (1 + args.slowdown * active / args.capacity))
            fin = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                   "choices": [{"index": 0, "delta": {}, "finish_reason": "length"}]}
            await resp.write(f"data: {json.dumps(fin)}\n\n".encode())
            await resp.write(b"data: [DONE]\n\n")
        finally:
            active -= 1
    await resp.write_eof()
    return resp

async def models(request):
    return web.json_response({"object": "list", "data": [{"id": "sim", "object": "model"}]})

async def health(request):
    return web.Response(text="ok")

app = web.Application()
app.router.add_post("/v1/chat/completions", chat)
app.router.add_get("/v1/models", models)
app.router.add_get("/health", health)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile gateway8.py
"""Logging gateway with an active timing watermark.
For every request it draws a secret fair coin b and, after the model has finished, holds back the last streamed chunk
(which carries [DONE]) for b * delta seconds. It logs: key, request id, arrival, first byte, true completion (backend
finished), end (last byte sent), status, bytes, b. Only the provider knows b and the true completion time."""
import asyncio, time, argparse, random, json
import aiohttp
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=9000)
ap.add_argument("--backend", default="http://127.0.0.1:8000")
ap.add_argument("--log", default="gateway_log.csv")
ap.add_argument("--delta", type=float, default=0.15)
ap.add_argument("--seed", type=int, default=12345)
ap.add_argument("--mode", default="holdback", choices=["holdback", "spread"])
args = ap.parse_args()
coin = random.Random(args.seed)
logf = open(args.log, "a", buffering=1)
if logf.tell() == 0:
    logf.write("key,rid,t_arrival,t_first,t_complete,t_end,status,nbytes,wm_bit\n")
session = None

async def on_start(app):
    global session
    session = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=None))

async def on_stop(app):
    await session.close(); logf.close()

async def proxy(request):
    key = request.headers.get("Authorization", "").replace("Bearer ", "").strip() or "anon"
    rid = request.headers.get("X-Req-Id", "")
    bit = coin.getrandbits(1)
    t_arr = time.time(); t_first = None; t_comp = None; nbytes = 0; status = 0; slept = 0.0
    body = await request.read()
    try:
        max_tok = max(1, int(json.loads(body).get("max_tokens") or 64))
    except Exception:
        max_tok = 64
    try:
        async with session.post(args.backend + request.path, data=body, headers={"Content-Type": "application/json"}) as r:
            status = r.status
            resp = web.StreamResponse(status=r.status, headers={"Content-Type": r.headers.get("Content-Type", "text/event-stream")})
            await resp.prepare(request)
            held = None
            async for chunk in r.content.iter_any():
                if t_first is None:
                    t_first = time.time()
                if args.mode == "spread":
                    # spread the delay over the stream: each token is held back delta / max_tokens
                    if bit and args.delta > 0:
                        dl = args.delta * chunk.count(b"data:") / max_tok
                        if dl > 0:
                            await asyncio.sleep(dl); slept += dl
                    await resp.write(chunk); nbytes += len(chunk)
                else:
                    if held is not None:
                        await resp.write(held); nbytes += len(held)
                    held = chunk
            if args.mode == "holdback":
                t_comp = time.time()
                if bit and args.delta > 0:
                    await asyncio.sleep(args.delta)
                if held is not None:
                    await resp.write(held); nbytes += len(held)
            await resp.write_eof()
            if args.mode == "spread":
                t_comp = time.time() - slept          # when the stream would have ended without the delay
    except (ConnectionResetError, aiohttp.ClientError, asyncio.CancelledError, RuntimeError):
        status = 499
        resp = web.Response(status=499)
    finally:
        t_end = time.time()
        logf.write(f"{key},{rid},{t_arr:.6f},{(t_first or 0):.6f},{(t_comp or t_end):.6f},{t_end:.6f},{status},{nbytes},{bit}\n")
    return resp

async def passthrough_get(request):
    async with session.get(args.backend + request.path) as r:
        return web.Response(status=r.status, body=await r.read(), content_type="application/json")

app = web.Application(client_max_size=10 * 1024 ** 2)
app.on_startup.append(on_start); app.on_cleanup.append(on_stop)
app.router.add_post("/v1/chat/completions", proxy)
app.router.add_get("/v1/models", passthrough_get)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile driver8.py
"""One client population per process (no shared event loop between populations).
usage: python driver6.py cfg.json POP
Writes labels_POP.json (key -> POP, ring position) and clientlog_POP.csv (rid, key, t_sched, t_send, t_done, ok).
Every request carries an X-Req-Id header so client-side and gateway-side timestamps can be joined."""
import asyncio, json, sys, time, itertools, random, math, logging, uuid
import numpy as np
from openai import AsyncOpenAI

cfg = json.load(open(sys.argv[1])); POP = sys.argv[2]; P = cfg["pops"][POP]
seed = cfg["seed"] * 1000 + sorted(cfg["pops"]).index(POP)
rng = np.random.default_rng(seed); random.seed(seed)
BASE = cfg["gateway"] + "/v1"; MODEL = cfg["model"]
T0 = cfg["t_start"]; T_END = T0 + cfg["run_s"]
labels = {}; stats = {"ok": 0, "err": 0}; err_types = {}; _clients = {}
clog = open(f"clientlog_{POP}.csv", "w", buffering=1); clog.write("rid,key,t_sched,t_send,t_done,ok,gap_last,gap_med\n")

def keys(n, prefix=None):
    prefix = prefix or POP
    ks = [f"{prefix}-{i:03d}" for i in range(P.get("key_offset", 0), P.get("key_offset", 0) + n)]
    for i, k in enumerate(ks): labels[k] = dict(pop=POP, pos=i)
    return ks

def client(key):
    if key not in _clients:
        _clients[key] = AsyncOpenAI(api_key=key, base_url=BASE, max_retries=0, timeout=300)
    return _clients[key]

WC = json.load(open(cfg["wc_file"])) if cfg.get("wc_file") else None
_pending_prompt = {}

def ntok(med=None):
    """output length; with WildChat lengths, also stages a matching prompt length for the next prompt() call"""
    if WC is not None and cfg.get("lengths") == "wildchat" and med is None:
        i = int(rng.integers(len(WC["out_tok"])))
        _pending_prompt["n"] = int(WC["prompt_tok"][i])
        return int(np.clip(WC["out_tok"][i], 16, cfg.get("max_out", 1024)))
    return int(np.clip(rng.lognormal(math.log(med or cfg["tok_med"]), cfg["tok_sig"]), 16, 512))

VOCAB = ["data", "model", "river", "light", "market", "theory", "system", "story", "garden", "signal", "planet", "music",
         "energy", "window", "people", "letter", "number", "city", "paper", "voice"]
def prompt():
    n = _pending_prompt.pop("n", 0)
    body = " ".join(random.choice(VOCAB) for _ in range(int(min(n, cfg.get("max_prompt", 1500)) / 1.3))) if n else ""
    return [{"role": "user", "content": f"Write a note about topic {random.randint(0, 10**6)}. {body}"}]

def note_err(ex):
    stats["err"] += 1
    k = f"{POP}:{type(ex).__name__}:{str(ex)[:80]}"; err_types[k] = err_types.get(k, 0) + 1

async def call(key, n, t_sched=None):
    rid = uuid.uuid4().hex[:16]; t_send = time.time(); ok = 0
    try:
        s = await client(key).chat.completions.create(model=MODEL, messages=prompt(), max_tokens=n, stream=True,
                                                      extra_body={"ignore_eos": True}, extra_headers={"X-Req-Id": rid})
        ts = []
        async for _ in s:
            ts.append(time.time())
        stats["ok"] += 1; ok = 1
    except Exception as ex:
        ts = []
        note_err(ex); await asyncio.sleep(0.5)
    g = np.diff(ts) if len(ts) > 2 else np.array([np.nan])
    clog.write(f"{rid},{key},{t_sched or t_send:.6f},{t_send:.6f},{time.time():.6f},{ok},{g[-1]:.6f},{np.median(g):.6f}\n")

def alive():
    return time.time() < T_END

async def start_delay(hi):
    await asyncio.sleep(max(0.0, T0 - time.time()) + rng.uniform(0, hi))

def gap():
    return rng.lognormal(math.log(cfg["client_gap_med"]), 0.5)

# ---------------------------------------------------------------- benign
async def human(key):
    h = cfg["human"]
    await start_delay(0); await asyncio.sleep(rng.exponential(h["gap_s"]))
    while alive():
        for _ in range(rng.geometric(1 / h["mean_req"])):
            if not alive(): return
            await call(key, ntok())
            await asyncio.sleep(rng.lognormal(math.log(h["think_med_s"]), h["think_sig"]))
        await asyncio.sleep(rng.exponential(h["gap_s"]))

async def human_wc(key):
    h = cfg["human"]
    await start_delay(0); await asyncio.sleep(rng.exponential(h["gap_s"]))
    while alive():
        n_turns = int(WC["session_turns"][rng.integers(len(WC["session_turns"]))]) if WC else int(rng.geometric(1 / h["mean_req"]))
        for _ in range(max(1, n_turns)):
            if not alive(): return
            await call(key, ntok())
            think = float(WC["think_s"][rng.integers(len(WC["think_s"]))]) if WC else rng.lognormal(math.log(h["think_med_s"]), h["think_sig"])
            await asyncio.sleep(think)
        await asyncio.sleep(rng.exponential(h["gap_s"]))

async def batch_client(key):
    await start_delay(5)
    while alive():
        await call(key, ntok()); await asyncio.sleep(gap())

# ---------------------------------------------------------------- dispatchers
async def rr_shared(ks, K, jitter=0.0):
    """K workers share ONE itertools.cycle over the keys (the common 'rotate keys' snippet)"""
    cyc = itertools.cycle(ks)
    async def worker():
        await start_delay(2)
        while alive():
            await call(next(cyc), ntok()); await asyncio.sleep(gap() + (rng.uniform(0, jitter) if jitter else 0.0))
    await asyncio.gather(*[worker() for _ in range(K)])

async def rr_hold(ks, K, H):
    """each worker keeps a key for H consecutive requests, then takes the next key from one shared cycle"""
    cyc = itertools.cycle(ks)
    async def worker():
        await start_delay(2)
        while alive():
            k = next(cyc)
            for _ in range(H):
                if not alive(): return
                await call(k, ntok()); await asyncio.sleep(gap())
    await asyncio.gather(*[worker() for _ in range(K)])

async def random_keys(ks, K, jitter=0.0):
    """K workers, each request on a uniformly random key (random.choice); optional uniform jitter"""
    async def worker():
        await start_delay(2)
        while alive():
            await call(ks[rng.integers(len(ks))], ntok()); await asyncio.sleep(gap() + (rng.uniform(0, jitter) if jitter else 0.0))
    await asyncio.gather(*[worker() for _ in range(K)])

async def rr_per_worker(ks, K):
    """each worker has its OWN cycle, started at a random key"""
    async def worker():
        i = int(rng.integers(len(ks)))
        await start_delay(2)
        while alive():
            await call(ks[i % len(ks)], ntok()); i += 1; await asyncio.sleep(gap())
    await asyncio.gather(*[worker() for _ in range(K)])

async def pinned(ks, K):
    """worker w always uses key w: K independent closed loops (the timing-only impossibility case)"""
    async def worker(k):
        await start_delay(2)
        while alive():
            await call(k, ntok()); await asyncio.sleep(gap())
    await asyncio.gather(*[worker(ks[w % len(ks)]) for w in range(K)])

async def litellm_router(ks, K, strategy):
    import litellm
    from litellm import Router
    litellm.suppress_debug_info = True
    if hasattr(litellm, "REPEATED_STREAMING_CHUNK_LIMIT"):
        litellm.REPEATED_STREAMING_CHUNK_LIMIT = 10 ** 9
    for n in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy"): logging.getLogger(n).setLevel(logging.ERROR)
    router = Router(model_list=[{"model_name": "teacher", "litellm_params": {"model": f"openai/{MODEL}", "api_base": BASE, "api_key": k}}
                                for k in ks], routing_strategy=strategy, num_retries=0)
    async def worker():
        await start_delay(2)
        while alive():
            rid = uuid.uuid4().hex[:16]; t_send = time.time(); ok = 0
            try:
                r = await router.acompletion(model="teacher", messages=prompt(), max_tokens=ntok(), stream=True,
                                             extra_body={"ignore_eos": True}, extra_headers={"X-Req-Id": rid})
                async for _ in r:
                    pass
                stats["ok"] += 1; ok = 1
            except Exception as ex:
                note_err(ex); await asyncio.sleep(0.5)
            clog.write(f"{rid},,{t_send:.6f},{t_send:.6f},{time.time():.6f},{ok},nan,nan\n")
            await asyncio.sleep(gap())
    await asyncio.gather(*[worker() for _ in range(K)])

async def agent_fleet(ks, n_agents):
    a = cfg["agent"]
    async def agent():
        await start_delay(5)
        while alive():
            key = ks[rng.integers(len(ks))]
            for _ in range(rng.geometric(1 / a["mean_steps"])):
                if not alive(): return
                await call(key, ntok(a["tok_med"]))
                await asyncio.sleep(rng.lognormal(math.log(a["tool_med_s"]), 0.6))
            await asyncio.sleep(rng.lognormal(math.log(1.0), 0.5))
    await asyncio.gather(*[agent() for _ in range(n_agents)])

async def open_loop(ks, lam, policy, on_s=None, off_s=None):
    """Relay of independent end users: Poisson arrivals on an ABSOLUTE schedule (no drift from event-loop lag).
    on_s/off_s: bursty relay (rate lam during exponential ON periods, silent during OFF periods)."""
    tasks = set(); t = T0; rr = 0
    on_end = T0 + (rng.exponential(on_s) if on_s else 1e18)
    while True:
        t += rng.exponential(1 / lam)
        if on_s and t >= on_end:
            t = on_end + rng.exponential(off_s); on_end = t + rng.exponential(on_s); continue
        if t >= T_END: break
        await asyncio.sleep(max(0.0, t - time.time()))
        if policy == "rr": k = ks[rr % len(ks)]; rr += 1
        else: k = ks[rng.integers(len(ks))]
        task = asyncio.create_task(call(k, ntok(), t_sched=t)); tasks.add(task); task.add_done_callback(tasks.discard)
    if tasks:
        await asyncio.wait(tasks, timeout=120)

async def main():
    kind = P["kind"]
    if kind == "human":
        jobs = [(human_wc if cfg.get("humans") == "wildchat" else human)(k) for k in keys(P["n"], prefix="hum")]
    elif kind == "batch":
        jobs = [batch_client(k) for k in keys(P["n"], prefix="bat")]
    elif kind == "rr_shared":
        jobs = [rr_shared(keys(P["M"]), P["K"], P.get("jitter", 0.0))]
    elif kind == "rr_hold":
        jobs = [rr_hold(keys(P["M"]), P["K"], P["H"])]
    elif kind == "random":
        jobs = [random_keys(keys(P["M"]), P["K"], P.get("jitter", 0.0))]
    elif kind == "rr_worker":
        jobs = [rr_per_worker(keys(P["M"]), P["K"])]
    elif kind == "pinned":
        jobs = [pinned(keys(P["M"]), P["K"])]
    elif kind == "litellm":
        jobs = [litellm_router(keys(P["M"]), P["K"], P["strategy"])]
    elif kind == "agents":
        jobs = [agent_fleet(keys(P["M"]), P["K"])]
    elif kind == "open_loop":
        jobs = [open_loop(keys(P["M"]), P["lam"], P["policy"], P.get("on_s"), P.get("off_s"))]
    else:
        raise ValueError(kind)
    json.dump(labels, open(f"labels_{POP}.json", "w"))
    async def progress():
        while alive():
            await asyncio.sleep(120)
            print(f"[{POP}] t={time.time() - T0:.0f}s ok={stats['ok']} err={stats['err']}", flush=True)
    await asyncio.gather(*jobs, progress())
    print(f"[{POP}] done ok={stats['ok']} err={stats['err']} errors={json.dumps(dict(sorted(err_types.items(), key=lambda kv: -kv[1])[:4]))}", flush=True)
    json.dump(dict(stats=stats, errors=err_types), open(f"driverstats_{POP}.json", "w"))

asyncio.run(main())


In [ ]:
# Cell 3 — client libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp", "openai", "litellm", "huggingface_hub", "pyarrow"], check=False)
import importlib; importlib.invalidate_caches()
import numpy as np, pandas as pd


In [ ]:
# Cell 4 — empirical human behaviour and request lengths from WildChat-1M (one shard)
WC_INFO = {}
try:
    import pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
    from huggingface_hub import HfApi, hf_hub_download
    repo = "allenai/WildChat-1M"
    name = sorted(f.path for f in HfApi().list_repo_tree(repo, repo_type="dataset", recursive=True) if getattr(f, "path", "").endswith(".parquet"))[0]
    t = pq.read_table(hf_hub_download(repo, name, repo_type="dataset"), columns=["conversation"])
    conv = t.column("conversation").combine_chunks(); flat = pc.list_flatten(conv)
    parent = pc.list_parent_indices(conv).to_numpy()
    role = flat.field("role").to_numpy(zero_copy_only=False)
    clen = pc.utf8_length(flat.field("content")).to_numpy(zero_copy_only=False).astype(float)
    ts = pc.cast(flat.field("timestamp"), pa.int64()).to_numpy(zero_copy_only=False).astype(float) / 1e6
    tu = pd.DataFrame(dict(conv=parent, role=role, clen=clen, ts=ts))
    us, asst = tu[tu.role == "user"].reset_index(drop=True), tu[tu.role == "assistant"].reset_index(drop=True)
    n = min(len(us), len(asst))
    prompt_tok = np.clip(us.clen.values[:n] / 4, 4, 4000); out_tok = np.clip(asst.clen.values[:n] / 4, 1, 4000)
    same_conv = us.conv.values[:n] == asst.conv.values[:n]
    a = asst.sort_values(["conv", "ts"]); gaps = np.diff(a.ts.values); nxt_out = a.clen.values[1:] / 4
    same = np.diff(a.conv.values) == 0
    think = gaps[same] - nxt_out[same] / 40.0                 # time between answers minus the next answer's generation (~40 tok/s)
    think = think[np.isfinite(think) & (think > 1) & (think < 900)]
    sess = us.groupby("conv").size().values
    rs = np.random.default_rng(0)
    pick = lambda x, k=40000: rs.choice(x, min(k, len(x)), replace=False)
    ii = pick(np.nonzero(same_conv)[0])
    wc = dict(prompt_tok=prompt_tok[ii].round().tolist(), out_tok=out_tok[ii].round().tolist(), think_s=pick(think).round(2).tolist(),
              session_turns=pick(sess).tolist())
    WC_INFO = dict(source=name, prompt_tok_median=float(np.median(prompt_tok)), out_tok_median=float(np.median(out_tok)),
                   out_tok_p90=float(np.quantile(out_tok, .9)), think_median_s=float(np.median(think)), think_p90_s=float(np.quantile(think, .9)),
                   session_turns_median=float(np.median(sess)), session_turns_mean=float(np.mean(sess)))
except Exception as ex:
    print("WildChat unavailable, falling back to parametric humans and lengths:", repr(ex)[:300])
    rs = np.random.default_rng(0)
    wc = dict(prompt_tok=np.clip(rs.lognormal(np.log(60), 1.0, 40000), 4, 1500).round().tolist(),
              out_tok=np.clip(rs.lognormal(np.log(250), 0.8, 40000), 16, 1024).round().tolist(),
              think_s=np.clip(rs.lognormal(np.log(25), 1.0, 40000), 1, 900).round(2).tolist(), session_turns=rs.geometric(1 / 3, 40000).tolist())
    WC_INFO = dict(source="fallback-parametric")
json.dump(wc, open("wc.json", "w"))
print(json.dumps(WC_INFO, indent=1))


In [ ]:
# Cell 5 — install vLLM (same routes as NB5–NB7) and define start/stop of the model server
def wait_http(url, timeout_s, proc=None):
    t = time.time()
    while time.time() - t < timeout_s:
        if proc is not None and proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(url, timeout=5); return True
        except Exception:
            time.sleep(5)
    return False

def tail(path, n=20):
    try: return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception: return ""

def gpu_present():
    try: return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception: return False

BACKEND = {"kind": None, "python": None, "install": {}}
if CFG["backend"] in ("auto", "vllm") and gpu_present():
    t_inst = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], capture_output=True, text=True)
    chk = subprocess.run([sys.executable, "-c", "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__)"], capture_output=True, text=True)
    if chk.returncode == 0:
        BACKEND.update(python=sys.executable)
    else:
        venv = "/tmp/vllm_env"; subprocess.run([sys.executable, "-m", "venv", venv]); py = os.path.join(venv, "bin", "python")
        subprocess.run([py, "-m", "pip", "install", "-q", "--upgrade", "pip"], capture_output=True)
        subprocess.run([py, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
        chk = subprocess.run([py, "-c", "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__)"], capture_output=True, text=True)
        if chk.returncode == 0: BACKEND.update(python=py)
    BACKEND["install"] = dict(minutes=round((time.time() - t_inst) / 60, 1), msg=(chk.stdout + chk.stderr).strip()[-300:])
    print(BACKEND["install"])
if BACKEND["python"] is None and CFG["backend"] == "vllm":
    raise RuntimeError("vLLM requested but not importable")

def start_backend(phase):
    """returns (proc, kind, args used). Tries the phase's argument sets in order, then eager mode."""
    if BACKEND["python"] is None:
        p = subprocess.Popen([sys.executable, "simserver.py", "--port", "8000"], stdout=open(f"sim_{phase}.log", "w"), stderr=subprocess.STDOUT)
        assert wait_http("http://127.0.0.1:8000/health", 60, p); return p, "sim", []
    base = [BACKEND["python"], "-m", "vllm.entrypoints.openai.api_server", "--model", CFG["model"], "--dtype", "half",
            "--max-model-len", "2048", "--max-num-seqs", "128", "--port", "8000"]
    for extra in CFG["vllm_extra"][phase] + [["--enforce-eager"]]:
        args = list(extra)
        if "--gpu-memory-utilization" not in args: args += ["--gpu-memory-utilization", "0.85"]
        p = subprocess.Popen(base + args, stdout=open(f"vllm_{phase}.log", "w"), stderr=subprocess.STDOUT)
        if wait_http("http://127.0.0.1:8000/v1/models", 900, p):
            return p, "vllm", args
        print("vLLM failed with", args, "\n", tail(f"vllm_{phase}.log", 12)); p.kill(); time.sleep(5)
    raise RuntimeError("vLLM could not start for phase " + phase)

def poll_metrics(stop, out):
    """vLLM Prometheus metrics every 10 s: KV-cache usage, preemptions, running and waiting requests"""
    while not stop.is_set():
        try:
            txt = urllib.request.urlopen("http://127.0.0.1:8000/metrics", timeout=5).read().decode()
            row = {"t": time.time()}
            for line in txt.splitlines():
                if line.startswith("vllm:") and any(s in line for s in ("cache_usage", "preempt", "num_requests_running", "num_requests_waiting")):
                    name = line.split("{")[0].split(" ")[0]
                    try: row[name] = row.get(name, 0.0) + float(line.rsplit(" ", 1)[1])
                    except ValueError: pass
            out.append(row)
        except Exception:
            pass
        stop.wait(10)


In [ ]:
# Cell 6 — run one phase: model server, watermarking gateway, every population in its own process
def run_phase(phase):
    rd = os.path.abspath(f"run_{phase}"); os.makedirs(rd, exist_ok=True)
    for f in glob.glob(os.path.join(rd, "*")): os.remove(f)
    proc, kind, args = start_backend(phase)
    gw = subprocess.Popen([sys.executable, os.path.abspath("gateway8.py"), "--port", "9000", "--backend", "http://127.0.0.1:8000",
                           "--log", os.path.join(rd, "gateway_log.csv"), "--delta", str(CFG["wm_delta"]), "--seed", str(4242 + ord(phase)),
                           "--mode", CFG["wm_mode"][phase]], stdout=open(os.path.join(rd, "gateway.out"), "w"), stderr=subprocess.STDOUT)
    assert wait_http(CFG["gateway"] + "/v1/models", 60, gw), tail(os.path.join(rd, "gateway.out"))
    model = CFG["model"] if kind == "vllm" else "sim"
    dcfg = dict(CFG, pops=CFG["pops"][phase], run_s=CFG["run_s"][phase], lengths=CFG["lengths"][phase], model=model,
                wc_file=os.path.abspath("wc.json"), t_start=time.time() + 40)
    json.dump(dcfg, open(os.path.join(rd, "driver_cfg.json"), "w"))
    stop, mets = threading.Event(), []
    th = threading.Thread(target=poll_metrics, args=(stop, mets), daemon=True); th.start()
    procs = {p: subprocess.Popen([sys.executable, os.path.abspath("driver8.py"), "driver_cfg.json", p], cwd=rd,
                                 stdout=open(os.path.join(rd, f"drv_{p}.out"), "w"), stderr=subprocess.STDOUT) for p in dcfg["pops"]}
    t_run = time.time()
    while any(pr.poll() is None for pr in procs.values()):
        time.sleep(120)
        n = sum(1 for _ in open(os.path.join(rd, "gateway_log.csv"))) - 1
        last = mets[-1] if mets else {}
        print(f"[{phase}] t={time.time() - dcfg['t_start']:.0f}s requests={n} running={sum(pr.poll() is None for pr in procs.values())} "
              f"metrics={ {k.split(':')[-1]: round(v, 3) for k, v in last.items() if k != 't'} }", flush=True)
    stop.set(); gw.terminate(); proc.terminate(); time.sleep(10)
    try: proc.kill()
    except Exception: pass
    tails = {}
    for p in procs:
        txt = open(os.path.join(rd, f"drv_{p}.out")).read().strip().splitlines()
        tails[p] = txt[-1][:200] if txt else ""
    m = pd.DataFrame(mets)
    msum = {}
    for c in m.columns:
        if c == "t": continue
        v = m[c].dropna()
        if len(v): msum[c.split(":")[-1]] = dict(max=round(float(v.max()), 3), p90=round(float(v.quantile(.9)), 3), last=round(float(v.iloc[-1]), 3))
    cfg_phase = dict(dcfg, expected=CFG["expected"], wm_mode=CFG["wm_mode"], warmup_s=CFG["warmup_s"], pops=CFG["pops"][phase])
    return dict(backend=kind, vllm_args=args, wall_min=round((time.time() - t_run) / 60, 1), driver_tail=tails, metrics=msum), cfg_phase, rd


In [ ]:
# Cell 7 — Phase A: stress run and frozen analysis
import importlib, occlib, fplib, nb8an
importlib.reload(occlib); importlib.reload(fplib); importlib.reload(nb8an)
print("fplib", fplib.__version__)
RES = {"wildchat": WC_INFO, "fplib": fplib.__version__}
runA, cfgA, rdA = run_phase("A")
RES["A_run"] = runA
RES["A"] = nb8an.analyze(rdA, cfgA, np.random.default_rng(CFG["seed"]), "A")
pd.read_csv(os.path.join(rdA, "gateway_log.csv")).to_csv("nb8_requests_A.csv", index=False)
print(json.dumps({k: RES["A"][k] for k in ("site", "watermark_human_null")}, indent=1, default=str))


In [ ]:
# Cell 8 — Phase B: evasion gradient
runB, cfgB, rdB = run_phase("B")
RES["B_run"] = runB
RES["B"] = nb8an.analyze(rdB, cfgB, np.random.default_rng(CFG["seed"] + 1), "B")
pd.read_csv(os.path.join(rdB, "gateway_log.csv")).to_csv("nb8_requests_B.csv", index=False)


In [ ]:
# Cell 9 — scorecard against the locked predictions, then the summary to paste
A, B = RES["A"], RES["B"]
P = lambda R, p, k="pool_w1": R["watermark"].get(p, {}).get(k, {}).get("p", float("nan"))
mA = runA["metrics"]
kv = max([v["max"] for k, v in mA.items() if "cache_usage" in k] or [float("nan")])
pre = max([v["max"] for k, v in mA.items() if "preempt" in k] or [0.0])
ok_k = lambda R, p: bool(R["pools"].get(p, {}).get("K_little") and R["pools"][p]["K_true"] and abs(R["pools"][p]["K_little"] / R["pools"][p]["K_true"] - 1) <= 0.25)
rec = lambda R, det, p, x: (R["site"][det]["recovery"].get(p) or {}).get(x) or 0
RES["scorecard"] = dict(
    A0_stress=dict(kv_cache_usage_max=kv, preemptions=pre, pass_=bool((kv >= 0.85) or (pre > 0))),
    A1_classification=f"{sum(r['correct'] for r in A['pools'].values())}/{len(A['pools'])} (pass if >= 10)",
    A1b_classification_v06=f"{sum(r['correct_v06'] for r in A['pools'].values())}/{len(A['pools'])} (pass if >= 10)",
    ring_periods_A={p: A["pools"][p]["ring_period_s"] for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr") if p in A["pools"]},
    A2b_spectral_v06=dict(**{p: A["site"]["spectral_v06"]["recovery"].get(p) for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr")},
                          humans=A["site"]["spectral_v06"]["humans"],
                          pass_=all(rec(A, "spectral_v06", p, "recall") >= 0.9 for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr")) and A["site"]["spectral_v06"]["humans"] <= 2),
    A2_spectral=dict(**{p: A["site"]["spectral"]["recovery"].get(p) for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr")},
                     humans=A["site"]["spectral"]["humans"],
                     pass_=all(rec(A, "spectral", p, "recall") >= 0.9 and rec(A, "spectral", p, "purity") >= 0.9 for p in ("P3_rrshared", "P5_rrjitter", "P8_relay_rr"))
                     and A["site"]["spectral"]["humans"] <= 2),
    A3_sideband=dict(P3=rec(A, "sideband_graph", "P3_rrshared", "recall"), P4=rec(A, "sideband_graph", "P4_rrworker", "recall"), humans=A["site"]["sideband_graph"]["humans"],
                     pass_=rec(A, "sideband_graph", "P3_rrshared", "recall") >= 0.9 and rec(A, "sideband_graph", "P4_rrworker", "recall") >= 0.9 and A["site"]["sideband_graph"]["humans"] <= 2),
    A4_random_relay_p=P(A, "P7_relay_random"), A4_pass=bool(P(A, "P7_relay_random") > 0.01),
    A5_nulls=dict(watermark=A["watermark_human_null"], R_frac_abs_z_gt3=A["null_humans_40"]["R"]["frac_abs_z_gt3"],
                  pass_=bool(A["watermark_human_null"]["n_p_below_0_01"] <= 2 and A["null_humans_40"]["R"]["frac_abs_z_gt3"] <= 0.05)),
    A6_K={p: ok_k(A, p) for p in ("P1_shuffle", "P2_least", "P3_rrshared", "P4_rrworker", "C_company")},
    B1_ring_z={p: (B["pools"][p]["ring_z"], B["pools"][p]["ring_z_multiscale"]) for p in ("E0_rr_J0", "E1_rr_J2", "E2_rr_J6", "E3_rr_J15", "E4_rr_J15_K24") if p in B["pools"]},
    B2_hold={p: dict(ring_z=B["pools"][p]["ring_z_multiscale"], z_R=B["pools"][p]["z_R"]) for p in ("E5_hold5", "E6_hold20") if p in B["pools"]},
    B3_classification_v06=f"{sum(r['correct_v06'] for r in B['pools'].values())}/{len(B['pools'])} (pass if >= 10)",
    B4_spread_watermark={p: P(B, p) for p in ("E0_rr_J0", "E7_shuffle_K8", "E8_shuffle_K18", "E2_rr_J6", "E3_rr_J15")},
    B5_visibility=dict(holdback=A.get("client_visibility_ms"), spread=B.get("client_visibility_ms")),
    B6_pinned_open=dict(E9=(B["pools"].get("E9_pinned", {}).get("predicted"), P(B, "E9_pinned")),
                        E10=(B["pools"].get("E10_open_steady", {}).get("predicted"), P(B, "E10_open_steady"))),
    B7_bursty=dict(predicted=B["pools"].get("E11_open_bursty", {}).get("predicted"), z_R=B["pools"].get("E11_open_bursty", {}).get("z_R")),
    cost_curve={p: dict(req_s_per_worker=B["pools"][p]["req_per_s_per_worker"], req_s=B["pools"][p]["req_per_s"], z_R=B["pools"][p]["z_R"],
                        ring_z=B["pools"][p]["ring_z_multiscale"], wm_p=P(B, p), cls=B["pools"][p]["predicted_v06"]) for p in B["pools"] if p.startswith("E")},
)
RES["runtime_min"] = round((time.time() - T_START) / 60, 1)
json.dump(RES, open("nb8_results.json", "w"), indent=1, default=str)
print(json.dumps(RES["scorecard"], indent=1, default=str))
print("===== PASTE FROM HERE =====")
print(json.dumps(RES, separators=(",", ":"), default=str))
print("===== PASTE TO HERE =====")
